# FMem Integration Demo

This notebook demonstrates how to use the `FMemClient` with `BaseAgent` for long-term memory.

In [ ]:
import os
import sys

# Add project root to path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.memory.fs_mem import FMemClient
from src.agent_blocks_hub.base_agent import create_base_agent
from src.llm_provider import get_llm
from langchain_core.messages import HumanMessage

## 1. Setup
Ensure you have `OPENAI_API_KEY` set.

In [ ]:
# Initialize LLM using the provider
llm = get_llm()

# Initialize Memory Client
task_dir = "./demo_memory"
client = FMemClient(task_dir=task_dir, llm=llm)

print(f"Memory initialized at: {task_dir}")

## 2. Create Agent
We create a `BaseAgent` and pass the memory client to `invoke`.

In [ ]:
agent = create_base_agent(llm=llm)

# Define a conversation state
state = {
    "messages": [
        HumanMessage(content="My name is Frank and I am working on an AI project.")
    ]
}

# Invoke agent WITH memory
response = agent.invoke(state, memory=client)

print("Agent Response:", response.content)

## 3. Save Context
After the turn, we can explicitly save important information to memory.

In [ ]:
# Update state with response
state["messages"].append(response)

# Save to memory
client.save_context(state["messages"])

print("Context saved to memory.")

## 4. Verify Memory Retrieval
Now let's ask a question that requires recalling the memory.

In [ ]:
new_state = {
    "messages": [
        HumanMessage(content="What is my name?")
    ]
}

# The agent should be able to answer because 'get_context' will retrieve the previous conversation/memory
response_2 = agent.invoke(new_state, memory=client)

print("Agent Response 2:", response_2.content)